# Preamble

In [1]:
# Running this script on a AWS Cloud SageMaker Notebook (ml.m5.2xlarge) with a volume size of 50 GB EBS

!python --version

Python 3.10.20


In [2]:
# Install Python packages that don't come pre-installed in AWS

!pip install scikit-learn
!pip install statsmodels

In [3]:
# Load required libraries

import boto3
import pandas as pd
import numpy as np
import re

import statsmodels.api as sm

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, average_precision_score

# Import Data

In [4]:
# Downloaded the CFPB compliants data from this website on June 16, 2026

# https://www.consumerfinance.gov/data-research/consumer-complaints/#get-the-data

In [5]:
# Import the CSV file of complaints locally OR from my AWS S3 bucket (where it is permanmently saved online)

# https://aletheia-public.s3.us-east-2.amazonaws.com/complaints_16Jun2026.csv

load_data_locally = True

if load_data_locally:
    df = pd.read_csv("complaints_in_just_2025.csv")  # just data from 2025
else:
    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket='aletheia-public', Key='complaints_16Jun2026.csv')  # all years of data
    df = pd.read_csv(obj['Body'])

In [6]:
# The full dataset of complaints has 15,896,496 observations
# The 2025 dataset of complaints has  5,443,005 observations

df.shape

(5443005, 16)

In [7]:
# Below is a data dictionary of all columns in the CFPB compliants data

# https://cfpb.github.io/api/ccdb/fields.html

In [8]:
# List the raw data types of all columns of the Pandas data frame

df.dtypes

Date received                   object
Product                         object
Sub-product                     object
Issue                           object
Sub-issue                       object
Consumer complaint narrative    object
Company public response         object
Company                         object
State                           object
ZIP code                        object
Tags                            object
Submitted via                   object
Date sent to company            object
Company response to consumer    object
Timely response?                object
Complaint ID                     int64
dtype: object

In [9]:
# Convert the date columns from a string object format to a datetime format with just a date (i.e. YYYY-MM-DD)

df['Date received'] = pd.to_datetime(df['Date received'].str[:10]).dt.normalize()
df['Date sent to company'] = pd.to_datetime(df['Date sent to company'].str[:10]).dt.normalize()

In [10]:
# Look at a few obs

df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2025-01-09,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account status incorrect,NaN,NaN,MOUNTAIN AMERICA FEDERAL CREDIT UNION,AZ,85301,NaN,Web,2025-01-27,Closed with explanation,Yes,11454251
1,2025-04-08,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,"To Whom It May Concern, I am filing a formal c...",NaN,MOHELA,NE,681XX,NaN,Web,2025-04-08,Closed with explanation,No,12856469
2,2025-04-15,Student loan,Federal student loan servicing,Struggling to repay your loan,"Problem with forgiveness, cancellation, or dis...",Struggling with a XXXX has significantly impac...,NaN,"Maximus Federal Services, Inc.",AL,35007,NaN,Web,2026-04-06,Closed with explanation,Yes,12989708
3,2025-04-18,Student loan,Federal student loan servicing,Dealing with your lender or servicer,Trouble with how payments are being handled,I have taken out parent loans on behalf of my ...,NaN,MOHELA,OH,43110,Servicemember,Web,2025-04-18,Closed with explanation,No,13062648
4,2025-05-05,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,NaN,NaN,MOHELA,OH,44224,NaN,Web,2025-05-05,Closed with explanation,No,13330457


In [11]:
# Group complaints by year received and count observations

df.groupby(df['Date received'].dt.year).size()

Date received
2025    5443005
dtype: int64

In [12]:
# Sample the data over different years and save as a CSV file for Alex

if not load_data_locally:
    df_sampled = df.sample(frac=0.25, random_state=42)
    df_sampled.to_csv('complaints_16Jun2026_sampled.csv', index=False)

In [13]:
# Filter complaints in just the year 2025 in order to reduce the size of the Pandas data frame

df = df[df['Date received'].dt.year == 2025]

In [14]:
# Confirm the count of observations in year 2025

df.shape

(5443005, 16)

In [15]:
# Count up the number of complaints with words in the consumer complaint narrative (i.e. not NaN) 
# Only 1,221,970 out of the 5,443,005 complaints in 2025 have words in the consumer complaint narrative

df[['Consumer complaint narrative']].count(axis=0)

Consumer complaint narrative    1221970
dtype: int64

In [16]:
df = df[df['Consumer complaint narrative'].notna()]

In [17]:
# Confirm the count of observations in the year 2025 with non-null values in the complaint narrative

df.shape

(1221970, 16)

# Explore Data

In [18]:
# Group complaints by product category and count observations

df.groupby(df['Product']).size()

Product
Checking or savings account                                 51447
Credit card                                                 41494
Credit reporting or other personal consumer reports        912758
Debt collection                                            100600
Debt or credit management                                    2641
Money transfer, virtual currency, or money service          64832
Mortgage                                                    14015
Payday loan, title loan, personal loan, or advance loan      7800
Prepaid card                                                 3822
Student loan                                                10881
Vehicle loan or lease                                       11680
dtype: int64

In [19]:
# Convert the Product categorical variable into a collectino of dummy variables
# use the category 'Credit reporting or other personal consumer reports' as the baseline category (because it is the most common)

df['Product'] = pd.Categorical(df['Product'], categories=['Credit reporting or other personal consumer reports',
                                                          'Checking or savings account', 
                                                          'Credit card', 
                                                          'Debt collection', 
                                                          'Debt or credit management', 
                                                          'Money transfer, virtual currency, or money service', 
                                                          'Mortgage', 
                                                          'Payday loan, title loan, personal loan, or advance loan', 
                                                          'Prepaid card', 
                                                          'Student loan', 
                                                          'Vehicle loan or lease'])

df = pd.get_dummies(df, columns=['Product'], drop_first=True, dtype=int)

In [20]:
# Group complaints by issue category and count observations

df.groupby(df['Issue']).size()

Issue
Advertising                                                       59
Advertising and marketing, including promotional offers         1987
Applying for a mortgage or refinancing an existing mortgage     1527
Attempts to collect debt not owed                              45773
Can't contact lender or servicer                                 134
                                                               ...  
Vehicle was repossessed or sold the vehicle                      105
Was approved for a loan, but didn't receive money                 15
Was approved for a loan, but didn't receive the money             75
Written notification about debt                                24189
Wrong amount charged or received                                 284
Length: 89, dtype: int64

In [21]:
# Group by the tags and count obsevations

df.groupby(df['Tags']).size()

Tags
Older American                   16263
Older American, Servicemember     4815
Servicemember                    51704
dtype: int64

In [22]:
# Recode this categorical feature of 'Tags' into two separate binary features

df['Older American'] = np.where(df['Tags'].isin(['Older American','Older American, Servicemember']), 1, 0)
df['Servicemember'] = np.where(df['Tags'].isin(['Servicemember','Older American, Servicemember']), 1, 0)

In [23]:
# Group by the company public response and count obsevations

df.groupby(df['Company public response']).size()

Company public response
Company believes complaint caused principally by actions of third party outside the control or direction of the company       297
Company believes complaint is the result of an isolated error                                                                 215
Company believes complaint relates to a discontinued policy or procedure                                                        4
Company believes complaint represents an opportunity for improvement to better serve consumers                                334
Company believes it acted appropriately as authorized by contract or law                                                    23150
Company believes the complaint is the result of a misunderstanding                                                            318
Company believes the complaint provided an opportunity to answer consumer's questions                                         954
Company can't verify or dispute the facts in the complaint        

In [24]:
# Group by company response to consumer and count obsevations

df.groupby(df['Company response to consumer']).size()

Company response to consumer
Closed with explanation            768024
Closed with monetary relief         15145
Closed with non-monetary relief    434484
In progress                           116
Untimely response                    4201
dtype: int64

In [25]:
# Collapse company response to consumer into a binary target: 

#   1 = complaint closed respectfully (with an explanation OR a monetary relief)
#   0 = complaint closed with no explanation

df['Target of Complaint Closed Respectfully'] = np.where(df['Company response to consumer'].isin(['Closed with explanation','Closed with monetary relief']), 1, 0)

In [26]:
# Group by target and count obsevations

df.groupby(df['Target of Complaint Closed Respectfully']).size()

Target of Complaint Closed Respectfully
0    438801
1    783169
dtype: int64

In [27]:
# Group by timely response to consumer complaints and count obsevations

df.groupby(df['Timely response?']).size()

Timely response?
No       10667
Yes    1211303
dtype: int64

In [28]:
# Group by how complaint was submitted via and count obsevations

df.groupby(df['Submitted via']).size()

Submitted via
Web    1221970
dtype: int64

In [29]:
# Convert the Product categorical variable into a collection of dummy variables 
# Use the category 'Web' as the baseline category (because it is the most common)

# Note that this is no longer a categorical feature in after our subsetting.

if False:

    df['Submitted via'] = pd.Categorical(df['Submitted via'], categories=['Web', 
                                                                          'Phone', 
                                                                          'Postal mail', 
                                                                          'Referral'])
    
    df = pd.get_dummies(df, columns=['Submitted via'], drop_first=True, dtype=int)

In [30]:
# Count up the number of unique companies of consumer complaints

df['Company'].nunique()

3224

In [31]:
# Find the top 20 companies with the most consumer complaints

df['Company'].value_counts().head(20)

Company
EQUIFAX, INC.                             315504
TRANSUNION INTERMEDIATE HOLDINGS, INC.    294058
Experian Information Solutions Inc.       268568
Block, Inc.                                38103
Early Warning Services, LLC                17665
CAPITAL ONE FINANCIAL CORPORATION          17160
JPMORGAN CHASE & CO.                       11795
NAVY FEDERAL CREDIT UNION                  10624
CITIBANK, N.A.                              9160
WELLS FARGO & COMPANY                       9047
BANK OF AMERICA, NATIONAL ASSOCIATION       8874
Resurgent Capital Services L.P.             8004
SYNCHRONY FINANCIAL                         6277
ENCORE CAPITAL GROUP INC.                   5684
Chime Financial Inc                         5648
MOHELA                                      5285
CL Holdings LLC                             5185
Portfolio Recovery Associates, LLC          5115
Paypal Holdings, Inc                        4809
AMERICAN EXPRESS COMPANY                    4766
Name: count,

In [32]:
# Print consumer complaints where the complaint narrative contains a specific keyword

keyword = 'Data Science'

with pd.option_context('display.max_colwidth', None):
    filtered_column = df.loc[df['Consumer complaint narrative'].str.contains(keyword, case=False, na=False), 'Consumer complaint narrative']
    print(filtered_column)

1399533    I enrolled in a one-year Postgraduate in Data Science program with XXXX XXXX XXXX XXXX from XX/XX/XXXX to XX/XX/XXXX. Toward the end of the program, I received a phone call from a XXXX XXXX representative named XXXX, who offered me an upgrade to a XXXX XXXX from XXXX de XXXX, stating it was free to explore and that there would be no financial obligation unless I chose to proceed. \n\nXXXX sent me a link to a site called Kodakco.com, assuring me it was just to check the upgrade offer or try it temporarily. Trusting XXXX XXXX as an educational provider, I clicked the link and followed instructions but I was misled into believing this was a no-obligation trial, not a financial agreement. I now realize I XXXX have unintentionally agreed to a {$1000.00} Klarna financing under false pretenses. \n\nSoon after, I was notified by Klarna of a {$1000.00} debt in my name linked to XXXX. There was no transparent checkout process or consent to financing terms. I took immediate action cont

# Feature Engineering

In [33]:
# Print out the current columns in the data frame

print(df.columns.tolist())

['Date received', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Complaint ID', 'Product_Checking or savings account', 'Product_Credit card', 'Product_Debt collection', 'Product_Debt or credit management', 'Product_Money transfer, virtual currency, or money service', 'Product_Mortgage', 'Product_Payday loan, title loan, personal loan, or advance loan', 'Product_Prepaid card', 'Product_Student loan', 'Product_Vehicle loan or lease', 'Older American', 'Servicemember', 'Target of Complaint Closed Respectfully']


In [34]:
# Count up the words in the consumer complaint narrative

df['complaint_word_count'] = df['Consumer complaint narrative'].str.split().str.len()

In [35]:
# Binary flag if the $ dollar sign in the consumer complaint narrative

df['dollar_sign'] = df['Consumer complaint narrative'].str.contains('$', na=False).astype(int)

In [36]:
# Binary flag if there are keywords related to fraud in the consumer complaint narrative

fraud_keywords = ['fraud', 'fraudulent', 'fake', 'forgery', 'trick', 'scam', 'swindle', 'hoax', 'scheme', 'sham']
fraud_pattern = '|'.join(fraud_keywords)

df['fraud_keyword'] = df['Consumer complaint narrative'].str.contains(fraud_pattern, case=False, na=False).astype(int)

In [37]:
# Binary flag if the company is a credit bureau 

df['credit_bureau'] = np.where(df['Company'].isin(['EQUIFAX, INC.',
                                                   'TRANSUNION INTERMEDIATE HOLDINGS, INC.', 
                                                   'Experian Information Solutions Inc.']), 1, 0)

In [38]:
# Binary flag if the company is a credit union

df['credit_union'] = df['Company'].str.contains('credit union', case=False, na=False).astype(int)

In [39]:
# Binary flag if the complaint comes from a Southern state

Southern_States = ['AL', 'AR', 'FL', 'GA', 'KY', 'LA', 'MS', 'NC', 'SC', 'TN', 'TX', 'VA', 'WV']

df['Southern_State'] = df['State'].isin(Southern_States).astype(int)

In [40]:
# Convert NaN to 0

df = df.fillna(0)

In [41]:
# Confirm that the shape of the data frame has not been affected

df.shape

(1221970, 34)

# Prepare for Modeling

In [42]:
# Print out the current columns in the data frame

print(df.columns.tolist())

['Date received', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Complaint ID', 'Product_Checking or savings account', 'Product_Credit card', 'Product_Debt collection', 'Product_Debt or credit management', 'Product_Money transfer, virtual currency, or money service', 'Product_Mortgage', 'Product_Payday loan, title loan, personal loan, or advance loan', 'Product_Prepaid card', 'Product_Student loan', 'Product_Vehicle loan or lease', 'Older American', 'Servicemember', 'Target of Complaint Closed Respectfully', 'complaint_word_count', 'dollar_sign', 'fraud_keyword', 'credit_bureau', 'credit_union', 'Southern_State']


In [43]:
# Subset just the relevant columns in the data frame

relevant_columns = ['Target of Complaint Closed Respectfully',
                    'Product_Checking or savings account',
                    'Product_Credit card',
                    'Product_Debt collection',
                    'Product_Debt or credit management',
                    'Product_Money transfer, virtual currency, or money service',
                    'Product_Mortgage',
                    'Product_Payday loan, title loan, personal loan, or advance loan',
                    'Product_Prepaid card',
                    'Product_Student loan',
                    'Product_Vehicle loan or lease',
                    'Older American',
                    'Servicemember',
                    'complaint_word_count',
                    'dollar_sign',
                    'fraud_keyword',
                    'credit_bureau',
                    'credit_union',
                    'Southern_State']

subset_df = df[relevant_columns]

In [44]:
# Clean up the column names for consistency by replacing commas and blank spaces with underscores

subset_df.columns = subset_df.columns.str.replace(",", "")
subset_df.columns = subset_df.columns.str.replace(" ", "_")

print(subset_df.columns.tolist())

['Target_of_Complaint_Closed_Respectfully', 'Product_Checking_or_savings_account', 'Product_Credit_card', 'Product_Debt_collection', 'Product_Debt_or_credit_management', 'Product_Money_transfer_virtual_currency_or_money_service', 'Product_Mortgage', 'Product_Payday_loan_title_loan_personal_loan_or_advance_loan', 'Product_Prepaid_card', 'Product_Student_loan', 'Product_Vehicle_loan_or_lease', 'Older_American', 'Servicemember', 'complaint_word_count', 'dollar_sign', 'fraud_keyword', 'credit_bureau', 'credit_union', 'Southern_State']


In [45]:
# Look at the data subset

subset_df.head()

,Target_of_Complaint_Closed_Respectfully,Product_Checking_or_savings_account,Product_Credit_card,Product_Debt_collection,Product_Debt_or_credit_management,Product_Money_transfer_virtual_currency_or_money_service,Product_Mortgage,Product_Payday_loan_title_loan_personal_loan_or_advance_loan,Product_Prepaid_card,Product_Student_loan,Product_Vehicle_loan_or_lease,Older_American,Servicemember,complaint_word_count,dollar_sign,fraud_keyword,credit_bureau,credit_union,Southern_State
1,1,0,0,0,0,0,0,0,0,0,0,0,0,173,1,0,0,0,0
2,1,0,0,0,0,0,0,0,0,1,0,0,0,98,1,0,0,0,1
3,1,0,0,0,0,0,0,0,0,1,0,0,1,292,1,0,0,0,0
6,1,0,0,0,0,0,0,0,0,0,0,0,0,26,1,0,0,0,0
7,1,0,0,0,0,0,0,0,0,0,0,0,1,356,1,0,0,0,0


In [46]:
# Separate feature vector (X) and target variable (y)

X = subset_df.drop('Target_of_Complaint_Closed_Respectfully', axis=1)
y = subset_df['Target_of_Complaint_Closed_Respectfully']

In [47]:
# Split the data into a training partition and a hold-out validation partition (70% training, 30% holdout)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Logistic Regression Model using Statsmodels

In [48]:
# Add a column of 1s to the feature set because statsmodels does not automatically add an intercept

X_train_with_constant = sm.add_constant(X_train)

In [49]:
# Fit a logistic regression model using statsmodel on the training data with robust standard errors
# https://blog.stata.com/2022/10/06/heteroskedasticity-robust-standard-errors-some-practical-considerations/

logistic_reg_stats_models = sm.Logit(y_train, X_train_with_constant).fit(cov_type='HC1')

Optimization terminated successfully.
         Current function value: 0.584269
         Iterations 8


In [50]:
# 2. View statistical summary of the results of fitting a Logistic regression model

print(logistic_reg_stats_models.summary())

                                      Logit Regression Results                                     
Dep. Variable:     Target_of_Complaint_Closed_Respectfully   No. Observations:               855379
Model:                                               Logit   Df Residuals:                   855361
Method:                                                MLE   Df Model:                           17
Date:                                     Sun, 12 Jul 2026   Pseudo R-squ.:                  0.1050
Time:                                             03:01:31   Log-Likelihood:            -4.9977e+05
converged:                                            True   LL-Null:                   -5.5839e+05
Covariance Type:                                       HC1   LLR p-value:                     0.000
                                                                   coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------

In [51]:
# Convert the Logistic regression model coefficients into Odds Ratios

# Extract log-odds coefficients and confidence intervals
df_coefficients = logistic_reg_stats_models.params.to_frame(name='Log-Odds')
df_confidence_intervals = logistic_reg_stats_models.conf_int()

# Combine them into a single summary DataFrame
odds_summary = pd.concat([df_coefficients, df_confidence_intervals], axis=1)
odds_summary.columns = ['Log-Odds', 'Lower CI (Log)', 'Upper CI (Log)']

# Exponentiate the entire DataFrame to calculate Odds Ratios
odds_summary['Odds Ratio'] = np.exp(odds_summary['Log-Odds'])
odds_summary['OR Lower CI'] = np.exp(odds_summary['Lower CI (Log)'])
odds_summary['OR Upper CI'] = np.exp(odds_summary['Upper CI (Log)'])

# Display the finalized odds ratios
odds_summary[['Odds Ratio', 'OR Lower CI', 'OR Upper CI']]

,Odds Ratio,OR Lower CI,OR Upper CI
Product_Checking_or_savings_account,4.959577,4.693313,5.240948
Product_Credit_card,1.800082,1.733524,1.869194
Product_Debt_collection,1.032492,1.010595,1.054864
Product_Debt_or_credit_management,2.437810,2.106199,2.821632
Product_Money_transfer_virtual_currency_or_money_service,16.608156,15.319692,18.004987
Product_Mortgage,7.578410,6.746508,8.512892
Product_Payday_loan_title_loan_personal_loan_or_advance_loan,5.128549,4.509753,5.832251
Product_Prepaid_card,4.617347,3.886575,5.485524
Product_Student_loan,1.108993,1.043412,1.178696
Product_Vehicle_loan_or_lease,3.695706,3.376023,4.045660


# Logistic Regression Model using Scikit-learn

In [52]:
# Scale features (Highly recommended for faster solver convergence) on just the Training partition to prevent data leakage

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [53]:
# Initialize and fit the Logistic Regression model

logistic_reg_sklearn_model = LogisticRegression(class_weight='balanced', solver='lbfgs', max_iter=100)
logistic_reg_sklearn_model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [54]:
# Make predictions

y_pred  = logistic_reg_sklearn_model.predict(X_test_scaled)              # Predict binary class (0 or 1)
y_probs = logistic_reg_sklearn_model.predict_proba(X_test_scaled)[:, 1]  # Predict raw probabilities

In [55]:
# Evaluate the performance

print("Logsitic Regression Model Evaluation on Hold-out Data:")
print("-" * 40)
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.2f}\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print(f"Area Under the Precision-Recall Curve (PR-AUC): {average_precision_score(y_test, y_probs):.2f}\n")

Logsitic Regression Model Evaluation on Hold-out Data:
----------------------------------------
Accuracy Score: 0.58

Confusion Matrix:
[[119768  12013]
 [142276  92534]]

Classification Report:
              precision    recall  f1-score   support

           0       0.46      0.91      0.61    131781
           1       0.89      0.39      0.55    234810

    accuracy                           0.58    366591
   macro avg       0.67      0.65      0.58    366591
weighted avg       0.73      0.58      0.57    366591

Area Under the Precision-Recall Curve (PR-AUC): 0.81



# Gradient Boosting Model using Scikit-learn

In [56]:
# Initialize and train the Gradient Boosting model on just the training partition

grad_boost_sklearn_model = HistGradientBoostingClassifier(class_weight='balanced', max_iter=100, learning_rate=0.1, max_depth=5, random_state=42)
grad_boost_sklearn_model.fit(X_train.values, y_train)

,loss,'log_loss'
,learning_rate,0.1
,max_iter,100
,max_leaf_nodes,31
,max_depth,5
,min_samples_leaf,20
,l2_regularization,0.0
,max_features,1.0
,max_bins,255
,categorical_features,'from_dtype'
,monotonic_cst,None


In [57]:
# Score and Evaluate on the hold-out validation partition

y_pred = grad_boost_sklearn_model.predict(X_test.values)               # Predict binary class (0 or 1)
y_probs = grad_boost_sklearn_model.predict_proba(X_test.values)[:, 1]  # Predict raw probabilities

In [58]:
# Evaluate the performance

print("Gradient Boosting Model Evaluation on Hold-out Data:")
print("-" * 40)
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.2f}\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print(f"Area Under the Precision-Recall Curve (PR-AUC): {average_precision_score(y_test, y_probs):.2f}\n")

Gradient Boosting Model Evaluation on Hold-out Data:
----------------------------------------
Accuracy Score: 0.58

Confusion Matrix:
[[118460  13321]
 [138921  95889]]

Classification Report:
              precision    recall  f1-score   support

           0       0.46      0.90      0.61    131781
           1       0.88      0.41      0.56    234810

    accuracy                           0.58    366591
   macro avg       0.67      0.65      0.58    366591
weighted avg       0.73      0.58      0.58    366591

Area Under the Precision-Recall Curve (PR-AUC): 0.82



# Random Forest Model using Scikit-learn

In [59]:
# Initialize and train the Random Forest model on just the training partition

random_forest_sklearn_model = RandomForestClassifier(class_weight='balanced', n_estimators=100, max_depth=5, random_state=42)
random_forest_sklearn_model.fit(X_train.values, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,5
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [60]:
# Score and Evaluate on the hold-out validation partition

y_pred = random_forest_sklearn_model.predict(X_test.values)               # Predict binary class (0 or 1)
y_probs = random_forest_sklearn_model.predict_proba(X_test.values)[:, 1]  # Predict raw probabilities

In [61]:
# Evaluate the performance

print("Random Forest Model Evaluation on Hold-out Data:")
print("-" * 40)
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.2f}\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print(f"Area Under the Precision-Recall Curve (PR-AUC): {average_precision_score(y_test, y_probs):.2f}\n")

Random Forest Model Evaluation on Hold-out Data:
----------------------------------------
Accuracy Score: 0.58

Confusion Matrix:
[[120188  11593]
 [142921  91889]]

Classification Report:
              precision    recall  f1-score   support

           0       0.46      0.91      0.61    131781
           1       0.89      0.39      0.54    234810

    accuracy                           0.58    366591
   macro avg       0.67      0.65      0.58    366591
weighted avg       0.73      0.58      0.57    366591

Area Under the Precision-Recall Curve (PR-AUC): 0.81

